# Final Project - Realized Volatility Timing




In [ ]:
from pathlib import Path
import sys
import matplotlib.pyplot as plt
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import pandas as pd
PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
from src.strategy.strategies import SHORT_1M_STRADDLE
from src.strategy.option_trade import DeltaHedgedOptionTrade, OptionTrade
from src.data_loader.option_loader import OptionLoader, extract_spot_from_options
from src.backtest.backtester import BacktesterBidAskFromData
from src.models.heston_kalman import HestonKalmanEstimator
from src.models.garch_benchmark import forecast_garch11_volatility
from src.signal.vol_signal import build_vol_signal, compute_strategy_iv_reference, rolling_realized_vol_benchmark, map_signal_to_trade_entries
from src.allocation.dynamic_allocation import tanh_allocation
pd.options.display.float_format = '{:.4f}'.format


## 1. Setup & Data

We work with SPY option data and US rates. The strategy legs are generated directly from the options database using the local project framework.


In [ ]:
start_date = pd.Timestamp('2020-01-02').to_pydatetime()
end_date = pd.Timestamp('2022-12-30').to_pydatetime()
ticker = 'SPY'

positions = DeltaHedgedOptionTrade.generate_trades(
    start_date=start_date,
    end_date=end_date,
    tickers=ticker,
    legs=SHORT_1M_STRADDLE,
)
positions.head()

## 2. Baseline Strategy

We first show that the carry strategy exists on its own before adding any volatility timing overlay.


In [ ]:
def compute_performance_metrics(nav_df):
    returns = nav_df['NAV'].pct_change().dropna()
    annualized_return = returns.mean() * 252
    annualized_volatility = returns.std(ddof=1) * (252 ** 0.5)

    q = int(min(21, max(1, len(returns) // 10))) if len(returns) > 1 else 1
    rho_sum = 0.0
    for k in range(1, q + 1):
        rho_k = returns.autocorr(lag=k)
        if pd.notna(rho_k):
            rho_sum += (1 - k / (q + 1)) * rho_k
    lo_scale = (1 + 2 * rho_sum) ** 0.5 if (1 + 2 * rho_sum) > 0 else float('nan')
    daily_sharpe = returns.mean() / returns.std(ddof=1) if returns.std(ddof=1) != 0 else float('nan')
    sharpe = daily_sharpe * (252 ** 0.5) / lo_scale if pd.notna(lo_scale) and lo_scale != 0 else float('nan')

    cumulative_nav = (1 + returns).cumprod()
    drawdown = cumulative_nav / cumulative_nav.cummax() - 1
    max_drawdown = drawdown.min()
    calmar = annualized_return / abs(max_drawdown) if max_drawdown != 0 else float('inf')
    return pd.Series({
        'annualized_return': annualized_return,
        'annualized_volatility': annualized_volatility,
        'sharpe': sharpe,
        'max_drawdown': max_drawdown,
        'calmar': calmar,
        'nav_end': nav_df['NAV'].iloc[-1],
    })

baseline = BacktesterBidAskFromData(positions).compute_backtest()
fig = go.Figure()
fig.add_trace(go.Scatter(x=baseline.nav.index, y=baseline.nav['NAV'], mode='lines', name='Baseline NAV'))
fig.update_layout(template='plotly_white', height=420, width=950, title='Baseline carry strategy NAV')
fig.update_xaxes(title='Date')
fig.update_yaxes(title='NAV')
fig.show()


In [ ]:
baseline_metrics = compute_performance_metrics(baseline.nav).to_frame('baseline')
baseline_metrics


### Baseline PnL Decomposition

We show the PnL decomposition once to make the carry mechanics visible without overloading the rest of the notebook.


In [ ]:
decomp_cols = ['pnl', 'model_pnl', 'tcost_pnl', 'delta_pnl', 'gamma_pnl', 'theta_pnl', 'vega_pnl', 'residual_pnl']
baseline_decomp = baseline.pnl[decomp_cols].copy()
baseline_decomp_summary = pd.DataFrame({
    'sum': baseline_decomp.sum(),
    'abs_sum': baseline_decomp.abs().sum(),
})
baseline_decomp_summary.loc['residual_share_abs_pnl', 'sum'] = baseline_decomp['residual_pnl'].abs().sum() / baseline_decomp['pnl'].abs().sum()
baseline_decomp_summary.loc['residual_share_abs_model_pnl', 'sum'] = baseline_decomp['residual_pnl'].abs().sum() / baseline_decomp['model_pnl'].abs().sum()
baseline_decomp_summary.loc['residual_share_greek_activity', 'sum'] = baseline_decomp['residual_pnl'].abs().sum() / baseline_decomp[['delta_pnl', 'gamma_pnl', 'theta_pnl', 'vega_pnl']].abs().sum().sum()
baseline_decomp_summary.loc['tcost_share_abs_pnl', 'sum'] = baseline_decomp['tcost_pnl'].abs().sum() / baseline_decomp['pnl'].abs().sum()
baseline_decomp_summary


In [ ]:
fig = go.Figure()
for col in ['pnl', 'model_pnl', 'tcost_pnl', 'delta_pnl', 'gamma_pnl', 'theta_pnl', 'vega_pnl', 'residual_pnl']:
    fig.add_trace(go.Scatter(x=baseline_decomp.index, y=baseline_decomp[col].cumsum(), mode='lines', name=col))
fig.update_layout(template='plotly_white', height=460, width=980, title='Baseline cumulative PnL decomposition')
fig.update_xaxes(title='Date')
fig.update_yaxes(title='Cumulative contribution')
fig.show()


## 3. Volatility Model (Heston UKF)

We estimate a latent variance process with a rolling UKF-Heston specification and compare it to simple realized-vol benchmarks.


In [ ]:
option_data = OptionLoader.load_data(start_date, end_date, process_kwargs={'ticker': ticker})
spot = extract_spot_from_options(option_data).set_index('date')['spot'].astype(float)
kalman_params = {
    'kappa': 0.9,
    'theta': 0.04,
    'xi': 0.5,
    'process_noise': 3e-5,
    'observation_noise': 3e-6,
    'alpha': 0.1,
    'rolling_window': 126,
    'recalibration_frequency': 10,
    'auto_calibrate': False,
}
estimator = HestonKalmanEstimator(**kalman_params)
sigma_hat = estimator.fit_transform(spot)
sigma_hat = estimator.add_horizon_forecast(sigma_hat, horizon_days=21, prefix='forecast')
rolling_vol = rolling_realized_vol_benchmark(spot, window=21)

log_returns = np.log(spot.astype(float)).diff()
future_squared_returns = pd.concat(
    [log_returns.pow(2).shift(-i) for i in range(1, 22)],
    axis=1,
)
future_realized_vol = np.sqrt(future_squared_returns.mean(axis=1) * 252)
future_realized_vol = future_realized_vol.rename('future_realized_vol').reset_index().rename(columns={'index': 'date'})


In [ ]:
model_compare = sigma_hat[['date', 'forecast_sigma_hat']].merge(future_realized_vol, on='date', how='inner').dropna()
fig = go.Figure()
fig.add_trace(go.Scatter(x=model_compare['date'], y=model_compare['forecast_sigma_hat'], mode='lines', name='UKF-Heston forecast'))
fig.add_trace(go.Scatter(x=model_compare['date'], y=model_compare['future_realized_vol'], mode='lines', name='Future realized vol'))
fig.update_layout(template='plotly_white', height=420, width=950, title='Forecast volatility vs future realized volatility')
fig.update_xaxes(title='Date')
fig.update_yaxes(title='Annualized volatility')
fig.show()


## 4. Signal Construction & Validation

The signal is built from the spread between implied volatility and the model forecast. This is the core of the project.


In [ ]:
iv_reference = compute_strategy_iv_reference(positions, option_data)
signal = build_vol_signal(iv_reference, sigma_hat, winsorize_quantiles=(0.01, 0.99))
scale_grid = [1.0, 2.0, 3.0]
allocation_grid = signal[["date", "vol_signal"]].copy()
for scale in scale_grid:
    allocation_grid[f'allocation_scale_{int(scale)}'] = tanh_allocation(
        signal['vol_signal'], scale=scale, leverage_cap=1.75, increasing=True
    )
chosen_scale = 1.5
signal['allocation_multiplier'] = allocation_grid[f'allocation_scale_{int(chosen_scale)}']
validated_signal = signal.merge(future_realized_vol, on='date', how='inner').dropna()
validated_signal['future_realized_minus_iv'] = validated_signal['future_realized_vol'] - validated_signal['iv_reference']
validated_signal['forecast_sigma_hat'] = validated_signal['sigma_hat']
validated_signal['signal_quantile'] = pd.qcut(
    validated_signal['vol_signal'], q=5, labels=False, duplicates='drop'
)


### Quantile Validation


In [ ]:
signal_quantile_analysis = validated_signal.groupby('signal_quantile', as_index=False)[['vol_signal', 'future_realized_minus_iv']].mean()
fig = go.Figure()
fig.add_trace(go.Bar(x=signal_quantile_analysis['signal_quantile'], y=signal_quantile_analysis['future_realized_minus_iv'], name='Average future RV'))
fig.update_layout(template='plotly_white', height=420, width=900, title='Quantile analysis of carry signal')
fig.update_xaxes(title='Signal quantile')
fig.update_yaxes(title='Average future realized vol minus IV')
fig.show()

### Signal Correlation


In [ ]:
signal_validation_corr = pd.DataFrame({
    'corr_with_future_realized_vol': {
        'vol_signal': validated_signal[['vol_signal', 'future_realized_vol']].corr().iloc[0, 1],
        'forecast_sigma_hat': validated_signal[['forecast_sigma_hat', 'future_realized_vol']].corr().iloc[0, 1],
    },
    'corr_with_future_realized_minus_iv': {
        'vol_signal': validated_signal[['vol_signal', 'future_realized_minus_iv']].corr().iloc[0, 1],
        'forecast_sigma_hat': validated_signal[['forecast_sigma_hat', 'future_realized_minus_iv']].corr().iloc[0, 1],
    },
})
signal_validation_corr


### Signal Distribution


In [ ]:
dynamic_positions = map_signal_to_trade_entries(positions, signal[['date', 'allocation_multiplier']], lag_business_days=1)
dynamic_positions['weight'] = dynamic_positions['weight'] * dynamic_positions['allocation_multiplier']
dynamic = BacktesterBidAskFromData(dynamic_positions[['date', 'option_id', 'entry_date', 'leg_name', 'weight', 'ticker']]).compute_backtest()

nav_compare = baseline.nav[['NAV']].rename(columns={'NAV': 'baseline'}).join(dynamic.nav[['NAV']].rename(columns={'NAV': 'dynamic'}), how='outer').ffill()
fig = go.Figure()
fig.add_trace(go.Scatter(x=nav_compare.index, y=nav_compare['baseline'], mode='lines', name='Baseline'))
fig.add_trace(go.Scatter(x=nav_compare.index, y=nav_compare['dynamic'], mode='lines', name='Dynamic'))
fig.update_layout(template='plotly_white', height=420, width=950, title='NAV comparison')
fig.update_xaxes(title='Date')
fig.update_yaxes(title='NAV')
fig.show()


metrics_table = pd.concat(
    {
        'baseline': compute_performance_metrics(baseline.nav),
        'dynamic_tanh': compute_performance_metrics(dynamic.nav),
        'dynamic_soft': compute_performance_metrics(dynamic_soft.nav),
    },
    axis=1,
)
metrics_table


In [ ]:
metrics_table = pd.concat(
    {
        'baseline': compute_performance_metrics(baseline.nav),
        'dynamic': compute_performance_metrics(dynamic.nav),
    },
    axis=1,
)
metrics_table


In [ ]:
fig = go.Figure()
for scale in scale_grid:
    fig.add_trace(go.Scatter(
        x=signal['vol_signal'],
        y=allocation_grid[f'allocation_scale_{int(scale)}'],
        mode='markers',
        name=f'scale={int(scale)}',
        marker=dict(size=7, opacity=0.45),
    ))
fig.update_layout(template='plotly_white', height=420, width=900, title='Signal vs allocation (IV - forecast spread)')
fig.update_xaxes(title='vol_signal')
fig.update_yaxes(title='allocation_multiplier')
fig.show()


In [ ]:
drawdown_compare = nav_compare.divide(nav_compare.cummax()).sub(1.0)
fig = go.Figure()
fig.add_trace(go.Scatter(x=drawdown_compare.index, y=drawdown_compare['baseline'], mode='lines', name='Baseline'))
fig.add_trace(go.Scatter(x=drawdown_compare.index, y=drawdown_compare['dynamic'], mode='lines', name='Dynamic'))
fig.update_layout(template='plotly_white', height=420, width=950, title='Drawdown comparison')
fig.update_xaxes(title='Date')
fig.update_yaxes(title='Drawdown')
fig.show()


### Hedged vs Unhedged Comparison

We also compare the same carry strategy with and without delta hedging to isolate the contribution of the hedge overlay.


In [ ]:

positions_unhedged = OptionTrade.generate_trades(
    start_date=start_date,
    end_date=end_date,
    tickers=ticker,
    legs=SHORT_1M_STRADDLE,
)
iv_reference_unhedged = compute_strategy_iv_reference(positions_unhedged, option_data)
signal_unhedged = build_vol_signal(
    iv_reference_unhedged,
    sigma_hat[['date', 'forecast_sigma_hat']],
    signal_definition='iv_minus_sigma',
    winsorize_quantiles=(0.05, 0.95),
)
signal_unhedged['allocation_multiplier'] = tanh_allocation(
    signal_unhedged['vol_signal'],
    scale=chosen_scale,
    leverage_cap=1.75,
    base=1.0,
    floor=0.25,
    increasing=True,
)
baseline_unhedged = BacktesterBidAskFromData(positions_unhedged).compute_backtest()
dynamic_positions_unhedged = map_signal_to_trade_entries(
    positions_unhedged,
    signal_unhedged[['date', 'allocation_multiplier']],
    lag_business_days=1,
)
dynamic_positions_unhedged['weight'] = dynamic_positions_unhedged['weight'] * dynamic_positions_unhedged['allocation_multiplier']
dynamic_unhedged = BacktesterBidAskFromData(
    dynamic_positions_unhedged[['date', 'option_id', 'entry_date', 'leg_name', 'weight', 'ticker']]
).compute_backtest()

hedge_metrics = pd.concat(
    {
        'baseline_hedged': compute_performance_metrics(baseline.nav),
        'dynamic_hedged': compute_performance_metrics(dynamic.nav),
        'baseline_unhedged': compute_performance_metrics(baseline_unhedged.nav),
        'dynamic_unhedged': compute_performance_metrics(dynamic_unhedged.nav),
    },
    axis=1,
)
hedge_metrics



In [ ]:

hedge_nav_compare = pd.concat(
    [
        baseline.nav['NAV'].rename('baseline_hedged'),
        dynamic.nav['NAV'].rename('dynamic_hedged'),
        baseline_unhedged.nav['NAV'].rename('baseline_unhedged'),
        dynamic_unhedged.nav['NAV'].rename('dynamic_unhedged'),
    ],
    axis=1,
).ffill()
fig = go.Figure()
for col in hedge_nav_compare.columns:
    fig.add_trace(go.Scatter(x=hedge_nav_compare.index, y=hedge_nav_compare[col], mode='lines', name=col))
fig.update_layout(template='plotly_white', height=460, width=980, title='Hedged vs unhedged NAV comparison')
fig.update_xaxes(title='Date')
fig.update_yaxes(title='NAV')
fig.show()



## 6. Benchmark: UKF vs Rolling Forecast

We compare the model-based forecast to a simple rolling-vol benchmark both as a forecast and as a trading input.


In [ ]:
benchmark_forecast = rolling_vol[['date', 'rolling_realized_vol']].copy()
benchmark_forecast['benchmark_sigma_hat'] = benchmark_forecast['rolling_realized_vol']
garch_forecast = forecast_garch11_volatility(
    np.log(spot.astype(float)).diff(),
    window=126,
    horizon_days=21,
    recalibration_frequency=21,
).rename(columns={'garch_sigma_hat': 'garch_sigma_hat'})
benchmark_signal = build_vol_signal(
    iv_reference,
    benchmark_forecast[['date', 'benchmark_sigma_hat']].rename(columns={'benchmark_sigma_hat': 'forecast_sigma_hat'}),
    signal_definition='iv_minus_sigma',
    winsorize_quantiles=(0.01, 0.99),
)
benchmark_signal['allocation_multiplier'] = tanh_allocation(
    benchmark_signal['vol_signal'],
    scale=1.5,
    leverage_cap=1.75,
    base=1.0,
    floor=0.25,
    increasing=True,
)
garch_signal = build_vol_signal(
    iv_reference,
    garch_forecast[['date', 'garch_sigma_hat']].rename(columns={'garch_sigma_hat': 'forecast_sigma_hat'}),
    signal_definition='iv_minus_sigma',
    winsorize_quantiles=(0.01, 0.99),
)
garch_signal['allocation_multiplier'] = tanh_allocation(
    garch_signal['vol_signal'],
    scale=1.5,
    leverage_cap=1.75,
    base=1.0,
    floor=0.25,
    increasing=True,
)
common_signal_start = max(signal['date'].min(), benchmark_signal['date'].min())
signal = signal.loc[signal['date'] >= common_signal_start].copy()
benchmark_signal = benchmark_signal.loc[benchmark_signal['date'] >= common_signal_start].copy()
garch_signal_start = garch_signal['date'].min()
garch_signal = garch_signal.loc[garch_signal['date'] >= garch_signal_start].copy()
benchmark_dynamic_positions = map_signal_to_trade_entries(positions, benchmark_signal[['date', 'allocation_multiplier']], lag_business_days=1)
benchmark_dynamic_positions['weight'] = benchmark_dynamic_positions['weight'] * benchmark_dynamic_positions['allocation_multiplier']
benchmark_dynamic = BacktesterBidAskFromData(benchmark_dynamic_positions[['date', 'option_id', 'entry_date', 'leg_name', 'weight', 'ticker']]).compute_backtest()
garch_dynamic_positions = map_signal_to_trade_entries(positions, garch_signal[['date', 'allocation_multiplier']], lag_business_days=1)
garch_dynamic_positions['weight'] = garch_dynamic_positions['weight'] * garch_dynamic_positions['allocation_multiplier']
garch_dynamic = BacktesterBidAskFromData(garch_dynamic_positions[['date', 'option_id', 'entry_date', 'leg_name', 'weight', 'ticker']]).compute_backtest()

future_forecast_compare = future_realized_vol.merge(
    sigma_hat[['date', 'forecast_sigma_hat']],
    on='date',
    how='inner',
).merge(
    benchmark_forecast[['date', 'benchmark_sigma_hat']],
    on='date',
    how='inner',
).merge(
    iv_reference[['date', 'iv_reference']],
    on='date',
    how='inner',
).dropna()
future_forecast_compare = future_forecast_compare.loc[future_forecast_compare['date'] >= common_signal_start].copy()
future_forecast_compare['future_realized_minus_iv'] = future_forecast_compare['future_realized_vol'] - future_forecast_compare['iv_reference']
future_forecast_compare['kalman_signal'] = future_forecast_compare['iv_reference'] - future_forecast_compare['forecast_sigma_hat']
future_forecast_compare['rolling_signal'] = future_forecast_compare['iv_reference'] - future_forecast_compare['benchmark_sigma_hat']
future_forecast_compare_garch = future_realized_vol.merge(
    garch_forecast[['date', 'garch_sigma_hat']],
    on='date',
    how='inner',
).merge(
    iv_reference[['date', 'iv_reference']],
    on='date',
    how='inner',
).dropna()
future_forecast_compare_garch = future_forecast_compare_garch.loc[future_forecast_compare_garch['date'] >= garch_signal_start].copy()
future_forecast_compare_garch['future_realized_minus_iv'] = future_forecast_compare_garch['future_realized_vol'] - future_forecast_compare_garch['iv_reference']
future_forecast_compare_garch['garch_signal'] = future_forecast_compare_garch['iv_reference'] - future_forecast_compare_garch['garch_sigma_hat']

forecast_benchmark_table = pd.DataFrame({
    'dynamic_sharpe': {
        'kalman': compute_performance_metrics(dynamic.nav)['sharpe'],
        'rolling_benchmark': compute_performance_metrics(benchmark_dynamic.nav)['sharpe'],
        'garch_benchmark': compute_performance_metrics(garch_dynamic.nav)['sharpe'],
    },
    'corr_forecast_future_rv': {
        'kalman': future_forecast_compare[['forecast_sigma_hat', 'future_realized_vol']].corr().iloc[0, 1],
        'rolling_benchmark': future_forecast_compare[['benchmark_sigma_hat', 'future_realized_vol']].corr().iloc[0, 1],
        'garch_benchmark': future_forecast_compare_garch[['garch_sigma_hat', 'future_realized_vol']].corr().iloc[0, 1],
    },
    'corr_signal_future_minus_iv': {
        'kalman': future_forecast_compare[['kalman_signal', 'future_realized_minus_iv']].corr().iloc[0, 1],
        'rolling_benchmark': future_forecast_compare[['rolling_signal', 'future_realized_minus_iv']].corr().iloc[0, 1],
        'garch_benchmark': future_forecast_compare_garch[['garch_signal', 'future_realized_minus_iv']].corr().iloc[0, 1],
    },
    'rmse_forecast_future_rv': {
        'kalman': float(((future_forecast_compare['forecast_sigma_hat'] - future_forecast_compare['future_realized_vol']) ** 2).mean() ** 0.5),
        'rolling_benchmark': float(((future_forecast_compare['benchmark_sigma_hat'] - future_forecast_compare['future_realized_vol']) ** 2).mean() ** 0.5),
        'garch_benchmark': float(((future_forecast_compare_garch['garch_sigma_hat'] - future_forecast_compare_garch['future_realized_vol']) ** 2).mean() ** 0.5),
    },
    'nav_end': {
        'kalman': dynamic.nav['NAV'].iloc[-1],
        'rolling_benchmark': benchmark_dynamic.nav['NAV'].iloc[-1],
        'garch_benchmark': garch_dynamic.nav['NAV'].iloc[-1],
    },
})
forecast_benchmark_table
